In [33]:
import random
random.seed(42)

# Modell 10 LSTM with Keras

### Was kann dieses Modell?

### Daten: ein Rennen
### Modelle: LSTM
### Training: sequentiell
### Vorhersage: Probs für alle Fahrer
### Problem: ???

## Imports

In [34]:
import fastf1
from fastf1 import plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# sklearn related imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.utils.class_weight import compute_sample_weight



project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.Feature_Engineer import FeatureEngineer
from utils.Metrics import Metrics
from utils.Models import Models
from utils.experiment_db import init_db
init_db()


pd.set_option('display.width', None)         # passt die Ausgabe-Breite an
pd.set_option('display.max_columns', None)   # zeigt alle Spalten
pd.set_option('display.max_rows', None)      # zeigt alle Reihen

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Rennen definieren

In [35]:
schedule = fastf1.get_event_schedule(2024, include_testing=False)
print(schedule["EventName"].tolist())

['Bahrain Grand Prix', 'Saudi Arabian Grand Prix', 'Australian Grand Prix', 'Japanese Grand Prix', 'Chinese Grand Prix', 'Miami Grand Prix', 'Emilia Romagna Grand Prix', 'Monaco Grand Prix', 'Canadian Grand Prix', 'Spanish Grand Prix', 'Austrian Grand Prix', 'British Grand Prix', 'Hungarian Grand Prix', 'Belgian Grand Prix', 'Dutch Grand Prix', 'Italian Grand Prix', 'Azerbaijan Grand Prix', 'Singapore Grand Prix', 'United States Grand Prix', 'Mexico City Grand Prix', 'São Paulo Grand Prix', 'Las Vegas Grand Prix', 'Qatar Grand Prix', 'Abu Dhabi Grand Prix']


In [36]:
YEAR = 2024
GRAND_PRIX = ['Belgian Grand Prix']
SESSION_TYPE = "R"

In [37]:
all_laps = []
for gp in GRAND_PRIX:
    try:
        session = fastf1.get_session(YEAR, gp, SESSION_TYPE)
        session.load(weather=True, laps=True, telemetry=True)
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps = laps.sort_values("Time")
        weather = weather.sort_values("Time")

        merged = pd.merge_asof(
                laps,
                weather,
                on="Time",
                direction="backward"
            )
        
        merged['GP'] = gp
        merged['Year'] = YEAR
        
        all_laps.append(merged)
        print(f"✔ Loaded {gp}")
        
    except Exception as e:
        print(f"Error loading {gp}: {e}")
        
data = pd.concat(all_laps, ignore_index=True)

core           INFO 	Loading data for Belgian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data


✔ Loaded Belgian Grand Prix


In [38]:
fe = FeatureEngineer()
merged = fe.create_features(data)

In [39]:
features = fe.choose_features(merged)
features

['gapAheadSeconds',
 's1_delta',
 's2_delta',
 's3_delta',
 'lap_pace_delta',
 'is_raining',
 'position_change',
 'compound_HARD',
 'compound_MEDIUM',
 'compound_SOFT',
 'status_1',
 'status_12']

In [40]:
merged = merged.sort_values(["GP", "Year", "LapNumber", "Driver"])

In [41]:
X = merged[features]
y = merged["pit_in_next_lap"]

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


X_train_raw = X_train_raw.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)
y_train_raw = y_train_raw.reset_index(drop=True)
y_test_raw = y_test_raw.reset_index(drop=True)

X_train_raw = X_train_raw.astype("float32")
X_test_raw = X_test_raw.astype("float32")


In [42]:
SEQ_LEN = 10

def create_sequences(features, labels, seq_len):
    X = []
    y = []
    for i in range(len(features) - seq_len):
        X.append(features[i:i + seq_len])
        y.append(labels[i + seq_len])
    return np.array(X), np.array(y) 


X_train, y_train = create_sequences(X_train_raw, y_train_raw, SEQ_LEN)
X_test, y_test = create_sequences(X_test_raw, y_test_raw, SEQ_LEN)


X_train = X_train.astype("float32")
X_test = X_test.astype("float32")
y_train = y_train.astype("float32")
y_test = y_test.astype("float32")


In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

In [44]:
from tensorflow.keras.metrics import AUC, Precision, Recall

model = Sequential([
    
    LSTM(64, input_shape=(SEQ_LEN, len(features))),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[
        AUC(name="auc"), 
        Precision(name="precision"), 
        Recall(name="recall")]
)


model.summary()

/Users/lukaskoch/anaconda3/envs/base_3_12/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))
print(class_weights)

{0.0: 0.5220820189274448, 1.0: 11.821428571428571}


In [46]:
X_train.astype('float32')
y_train.astype('float32')

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    class_weight= class_weights
)

Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - auc: 0.4239 - loss: 0.7083 - precision: 0.0326 - recall: 0.4000 - val_auc: 0.3698 - val_loss: 0.7043 - val_precision: 0.0244 - val_recall: 0.3333
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.6720 - loss: 0.6676 - precision: 0.0614 - recall: 0.5600 - val_auc: 0.3438 - val_loss: 0.6284 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7134 - loss: 0.6462 - precision: 0.0930 - recall: 0.4800 - val_auc: 0.3359 - val_loss: 0.5635 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7301 - loss: 0.6264 - precision: 0.0794 - recall: 0.6000 - val_auc: 0.2969 - val_loss: 0.6394 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.7287 - loss: 0.6156 - precision: 0.0843 - recall: 0.5600 - val_auc: 0.2995 - val_loss: 0.6185 - val_precision: 0.0000e+

In [47]:
y_pred_prob = model.predict(X_test)

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/stepWARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x313849260> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step


In [48]:
from utils.experiment_db import log_or_update_metrics

metrics = Metrics.calculate_metrics(f"approach_10_LSTM", y_test, y_pred_prob)
Metrics.print_metrics(metrics)
    
    
log_or_update_metrics(
    model_name=f"approach_10_{LSTM}",
    metrics={
        **metrics
    }
)

   ✔ Brier-Score:              0.0717 -> optimal 0.0
   ✔ Log-Loss:                 0.5082 -> optimal 0.0
   ✔ ROC-AUC:                  0.4129 -> optimal 1.0
   ✔ PR-AUC:                   0.0401 -> optimal 1.0
